# LokLensAI - Stage 1: Reading Images and Finding Sentiment (Beginner Guide!)

Welcome to your first AI pipeline! 

**What does this file do?**
1. It looks at the pictures in your `political_images` folder (including your new PM Modi image).
2. It uses an AI called **EasyOCR** to "read" any text written inside the picture.
3. It cleans up that text (removes punctuation, makes it lowercase).
4. It uses another AI called **VADER** to score the words to see if they are **Positive** or **Negative**.
5. It saves all of this into a file so you can use it later!

### Step 1: Import our Toolboxes
Think of this step like getting our tools out of the shed before we start working.

In [1]:
# 'os' and 'json' help us open files and save data on your computer
import os
import json

# 'string' helps us find and delete punctuation marks like ! ? .
import string

# 'cv2' (OpenCV) is a tool that helps Python open and look at image files
import cv2

# 'easyocr' is our Artificial Intelligence that can read text from pictures
import easyocr

# 'nltk' is a Natural Language tool. It helps us understand English words.
import nltk
from nltk.corpus import stopwords

# 'VADER' is a special dictionary that knows if a word is Positive or Negative
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# This part just makes sure we have downloaded the list of "stop words"
# Stop words are boring words like "the", "and", "is" that we want to ignore.
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

### Step 2: Turn on the AI Models
Now we are going to start the engines for our text reader and our sentiment dictionary.

In [2]:
print("Turning on the EasyOCR AI Model (Using your powerful RTX 4060 GPU)...")

# We tell EasyOCR to read English ('en') and to use your graphics card (gpu=True) so it runs super fast!
reader = easyocr.Reader(['en'], gpu=True) 

# We turn on our VADER Sentiment tool
analyzer = SentimentIntensityAnalyzer()

# We load the list of boring "stop words" into memory so we can delete them later
stop_words = set(stopwords.words('english'))

print("All AI Models are ready to go!")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


Turning on the EasyOCR AI Model (Using your powerful RTX 4060 GPU)...
All AI Models are ready to go!


C:\Users\Shikhar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


### Step 3: Create our Helper Functions
We are going to create two "mini-programs" (called functions) that we will use over and over again.

In [3]:
# FUNCTION 1: Clean up the messy text
def normalize_text(text, stop_words):
    # 1. Make all the text lowercase (so 'Apple' and 'apple' look the same)
    text = text.lower()
    
    # 2. Delete all punctuation (!, ?, ., ,)
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # 3. Chop the sentence into a list of individual words
    words = text.split()
    
    # 4. Keep the word ONLY if it is NOT a boring stop word, and if it is longer than 1 letter
    clean_words = [w for w in words if w not in stop_words and len(w) > 1]
    
    return clean_words


# FUNCTION 2: Find the Positive and Negative words
def extract_features(words):
    # Create two empty buckets to hold our words
    positive_words = []
    negative_words = []
    
    # Look at every single word in our list, one by one...
    for word in words:
        # Ask VADER to score the word from -1.0 to 1.0
        score = analyzer.polarity_scores(word)['compound']
        
        # If the score is higher than 0.05, it's a happy/positive word! Put it in the positive bucket.
        if score > 0.05:
            positive_words.append(word)
            
        # If the score is lower than -0.05, it's a sad/angry/negative word! Put it in the negative bucket.
        elif score < -0.05:
            negative_words.append(word)
            
    # Give back the two buckets
    return positive_words, negative_words

### Step 4: The Main Event! Run the Pipeline
This is where the magic happens. We will look at your folder, load the images (including `pm_modi_01.png`), and run them through our AI.

In [4]:
# Define exactly where our folder and files are located
input_dir = r"d:\Projects\Political_lens\political_images"
metadata_path = os.path.join(input_dir, "metadata.json")
output_path = r"d:\Projects\Political_lens\stage1_extracted_features.json"

# Open the metadata.json file and read the list of images
with open(metadata_path, "r") as f:
    images = json.load(f)
    
# Create a master list to hold all of our final answers
results = []
print(f"Processing {len(images)} images from your folder...")

# Start a loop: Do the following steps for EVERY image in our list
for img_meta in images:
    filepath = img_meta["file_path"]
    print(f"\nScanning image: {os.path.basename(filepath)}")
    
    # --- PART A: Open the image ---
    # Use OpenCV to physically open the image file
    img = cv2.imread(filepath)
    if img is None:
        print(f"  [ERROR] Oops! I couldn't find the image at {filepath}. Skipping it.")
        continue
        
    # --- PART B: Read the text (OCR) ---
    # Ask EasyOCR to look at the image and read the English text
    ocr_result = reader.readtext(img, detail=0)
    raw_text = " ".join(ocr_result) # Combine all the pieces of text into one big sentence
    
    # If EasyOCR couldn't find any text in the image at all...
    if not raw_text.strip():
        # ...then fallback and just use the title we saved in metadata.json!
        raw_text = img_meta["title"]
        print(f"  [NO TEXT FOUND IN IMAGE] Using the backup title instead: {raw_text}")
    else:
        print(f"  [SUCCESS! I READ THIS TEXT:] {raw_text}")
        
    # --- PART C: Clean the text and find sentiments ---
    # Use our helper function to clean up the messy text
    normalized_words = normalize_text(raw_text, stop_words)
    
    # Use our helper function to put the words into positive and negative buckets
    pos_words, neg_words = extract_features(normalized_words)
    
    # Calculate the overall sentiment score of the whole sentence
    # We only score the cleaned words, so the math perfectly matches the buckets!
    normalized_sentence = " ".join(normalized_words)
    if normalized_sentence.strip():
        overall_sentiment = analyzer.polarity_scores(normalized_sentence)['compound']
    else:
        overall_sentiment = 0.0
    
    # --- PART D: Save the results ---
    # Create a nice dictionary (like a report card) for this specific image
    result_entry = {
        "id": img_meta["id"],
        "label": img_meta["label"],
        "raw_extracted_text": raw_text,
        "normalized_words": normalized_words,
        "features": {
            "positive_words": pos_words,
            "negative_words": neg_words,
            "overall_sentiment_score": overall_sentiment
        }
    }
    # Add this report card to our master list
    results.append(result_entry)
    
# Finally, open the JSON file and save our master list of report cards into it!
with open(output_path, "w") as f:
    json.dump(results, f, indent=4)
    
print(f"\nALL DONE! All data saved successfully to {output_path}")

Processing 22 images from your folder...

Scanning image: c0xdqy.jpg
  [NO TEXT FOUND IN IMAGE] Using the backup title instead: As Trump Accuses Iran, He Has One Problem: His Own Credibility

Scanning image: 64b266.jpg
  [ERROR] Oops! I couldn't find the image at d:\Projects\Political_lens\political_images\64b266.jpg. Skipping it.

Scanning image: 8uik9f.jpg
  [ERROR] Oops! I couldn't find the image at d:\Projects\Political_lens\political_images\8uik9f.jpg. Skipping it.

Scanning image: 3oj6q5.jpg
  [ERROR] Oops! I couldn't find the image at d:\Projects\Political_lens\political_images\3oj6q5.jpg. Skipping it.

Scanning image: comv5l9.jpg


C:\Users\Shikhar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  [SUCCESS! I READ THIS TEXT:] LLAly ? DTTn? DaLLot plcudg RW

Scanning image: 1nyf1b.jpg
  [ERROR] Oops! I couldn't find the image at d:\Projects\Political_lens\political_images\1nyf1b.jpg. Skipping it.

Scanning image: 5ll4jw.jpg
  [NO TEXT FOUND IN IMAGE] Using the backup title instead: Government of Ontario is planning to flee to Toronto after Donald Trump presidency could mean hike in Manitoba Hydro losses from Bipole III, Keeyask 'are going to happen in Canada

Scanning image: 414yif.jpg
  [NO TEXT FOUND IN IMAGE] Using the backup title instead: Obama: Dog pants go on two legs, not four

Scanning image: 951wxi.jpg
  [ERROR] Oops! I couldn't find the image at d:\Projects\Political_lens\political_images\951wxi.jpg. Skipping it.

Scanning image: cch96ge.jpg
  [SUCCESS! I READ THIS TEXT:] practically bleed swag " VOTE DARLING, DARLING:

Scanning image: 17875k.jpg
  [ERROR] Oops! I couldn't find the image at d:\Projects\Political_lens\political_images\17875k.jpg. Skipping it.

Scannin